# LingueTech — Geometry of Truth on PRM800K

**Pipeline:**
1. Get PRM800K data
2. Prepare dataset (parse steps, derive concept labels)
3. Extract Llama-2-7b hidden states per layer
4. Logistic regression probing per layer
5. Mass-Mean Probing (all concepts)
6. PCA visualization
7. INSIDE / EigenScore

**Requirements:** Kaggle GPU (T4 recommended), HuggingFace token for Llama-2.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE — enable GPU in Settings')
print('CUDA version:', torch.version.cuda)

In [ ]:
# install missing deps (Kaggle has most, but double-check)
!pip install -q transformers accelerate pyarrow

In [ ]:
import os

# --- set your HuggingFace token ---
# option A: Kaggle Secrets (recommended)
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

# option B: paste directly (not recommended for public notebooks)
# os.environ['HF_TOKEN'] = 'hf_...'

# store model weights in /kaggle/working/ (20 GB quota)
os.environ['HF_CACHE_DIR'] = '/kaggle/working/hf_cache'

In [ ]:
# clone your code from GitHub
!git clone https://github.com/Daniil161-lang/LingueTech.git
%cd LingueTech

In [ ]:
# get PRM800K dataset
# repo structure after clone: prm800k/prm800k/data/phase2_train.jsonl
!git lfs install
!git clone https://github.com/openai/prm800k.git
!mkdir -p data
!cp prm800k/prm800k/data/phase2_train.jsonl data/phase2_train.jsonl
!echo 'PRM800K lines:' $(wc -l < data/phase2_train.jsonl)

In [ ]:
# step 1: parse PRM800K → dataset.parquet (+ concept columns)
!python prepare_dataset.py

In [ ]:
import pandas as pd
df = pd.read_parquet('data/dataset.parquet')
print('shape:', df.shape)
print('columns:', df.columns.tolist())
print('labels:', df['label'].value_counts().to_dict())
df.head(3)

In [ ]:
# step 2: extract hidden states — saves layers/layer{1..33}.csv
# ~30-60 min on T4 for 5000 samples; checkpoints every 50 batches
!python extract_hidden_states.py

In [ ]:
# step 3: logistic regression probing per layer
!python probing.py

In [ ]:
import pandas as pd
results = pd.read_csv('data/probing_results.csv')
print(results[['layer','accuracy_mean','f1_mean','roc_auc_mean']].to_string(index=False))

In [ ]:
# step 4: mass-mean probing for all concepts (label + epistemic markers)
!python mass_mean_probe.py --all

In [ ]:
# step 5: PCA visualization + ROC-AUC curves
!python pca_viz.py

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

for fname in sorted(os.listdir('plots')):
    img = mpimg.imread(f'plots/{fname}')
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(fname)
    plt.show()

In [ ]:
# step 6: INSIDE / EigenScore
# generates N continuations per step, measures consistency via eigenvalue entropy
# ~2-4 hours for 5000 samples; use --eval-only to skip generation if cached
!python eigenscore.py

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

es = pd.read_csv('data/eigenscore_results.csv')
fig, ax = plt.subplots(figsize=(7, 4))
for label, name, color in [(1, 'correct', 'steelblue'), (0, 'wrong', 'salmon')]:
    ax.hist(es[es['label'] == label]['eigenscore'], bins=40, alpha=0.6, label=name, color=color)
ax.set_xlabel('EigenScore (higher = less consistent = more uncertain)')
ax.set_ylabel('count')
ax.set_title('INSIDE EigenScore distribution')
ax.legend()
plt.tight_layout()
plt.show()